# 第一段：导入库和定义辅助函数

In [1]:
import os
import cv2
import insightface
import numpy as np
from sklearn import preprocessing
import socket
import base64
import struct
import pickle
import warnings

warnings.filterwarnings("ignore")

def find_most_common_string(string_list):
    string_counts = {}
    for s in string_list:
        if s in string_counts:
            string_counts[s] += 1
        else:
            string_counts[s] = 1
    most_common_string = max(string_counts, key=string_counts.get)
    total_count = sum(string_counts.values())
    percentage = string_counts[most_common_string] / total_count
    return [most_common_string, percentage]

def receive_image(sock, img_size):
    img_data = b''
    while len(img_data) < img_size:
        packet = sock.recv(4096)
        if not packet:
            return None
        img_data += packet
    base64_img = img_data.decode().split(',')[1]
    decoded_img = base64.b64decode(base64_img)
    img = cv2.imdecode(np.frombuffer(decoded_img, np.uint8), cv2.IMREAD_COLOR)
    return img

def is_turning(kpss):
    outx = []
    outy = []
    for kps in kpss:
        left_eye = kps[0]
        right_eye = kps[1]
        outx.append(right_eye[0] - left_eye[0])
        outy.append(left_eye[1] - right_eye[1])
    xmin = min(outx)
    xmax = max(outx)
    ymin = min(outy)
    ymax = max(outy)
    out = [xmin, xmax, ymin, ymax]
    a = abs(out[1] - out[0]) / (abs(out[1]) + abs(out[0]) + 0.001)
    b = abs(out[3] - out[2]) / (abs(out[3]) + abs(out[2]) + 0.001)
    return [a, b]


ERROR:albumentations.check_version:Error fetching version info
Traceback (most recent call last):
  File "c:\Users\27641\anaconda3\Lib\site-packages\albumentations\check_version.py", line 32, in fetch_version_info
    data = response.read()
           ^^^^^^^^^^^^^^^
  File "c:\Users\27641\anaconda3\Lib\http\client.py", line 489, in read
    s = self._safe_read(self.length)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\27641\anaconda3\Lib\http\client.py", line 638, in _safe_read
    data = self.fp.read(amt)
           ^^^^^^^^^^^^^^^^^
  File "c:\Users\27641\anaconda3\Lib\socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\27641\anaconda3\Lib\ssl.py", line 1314, in recv_into
    return self.read(nbytes, buffer)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\27641\anaconda3\Lib\ssl.py", line 1166, in read
    return self._sslobj.read(len, buffer)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TimeoutError: Th

# 第二段：定义人脸识别类

In [2]:
class FaceRecognition:
    def __init__(self, gpu_id=0, face_db='face_db', threshold=1.24, det_thresh=0.50, det_size=(640, 640)):
        self.gpu_id = gpu_id
        self.face_db = face_db
        self.threshold = threshold
        self.det_thresh = det_thresh
        self.det_size = det_size

        self.model = insightface.app.FaceAnalysis(root='./', allowed_modules=None, providers=['CUDAExecutionProvider'])
        self.model.prepare(ctx_id=self.gpu_id, det_thresh=self.det_thresh, det_size=self.det_size)
        self.faces_embedding = list()
        self.load_faces(self.face_db)

    def load_faces(self, face_db_path):
        if not os.path.exists(face_db_path):
            os.makedirs(face_db_path)
        for root, dirs, files in os.walk(face_db_path):
            for file in files:
                input_image = cv2.imdecode(np.fromfile(os.path.join(root, file), dtype=np.uint8), 1)
                if input_image is None:
                    print(f"Warning: Unable to read {file}. Skipping this file.")
                    continue
                user_name = file.split(".")[0]
                faces = self.model.get(input_image)
                if len(faces) == 0:
                    print(f"Warning: No face detected in {file}. Skipping this file.")
                    continue
                face = faces[0]
                embedding = np.array(face.embedding).reshape((1, -1))
                embedding = preprocessing.normalize(embedding)
                self.faces_embedding.append({
                    "user_name": user_name,
                    "feature": embedding
                })

    def recognition(self, image):
        faces = self.model.get(image)
        results = list()
        for face in faces:
            embedding = np.array(face.embedding).reshape((1, -1))
            embedding = preprocessing.normalize(embedding)
            user_name = "unknown"
            for com_face in self.faces_embedding:
                r = self.feature_compare(embedding, com_face["feature"], self.threshold)
                if r:
                    user_name = com_face["user_name"]
            results.append((user_name, face.bbox))
        return results

    @staticmethod
    def feature_compare(feature1, feature2, threshold):
        diff = np.subtract(feature1, feature2)
        dist = np.sum(np.square(diff), 1)
        return dist < threshold

    def register(self, image, user_name):
        faces = self.model.get(image)
        if len(faces) != 1:
            return '图片检测不到人脸'
        embedding = np.array(faces[0].embedding).reshape((1, -1))
        embedding = preprocessing.normalize(embedding)
        is_exits = False
        for com_face in self.faces_embedding:
            r = self.feature_compare(embedding, com_face["feature"], self.threshold)
            if r:
                is_exits = True
        if is_exits:
            return '该用户已存在'
        cv2.imencode('.jpg', image)[1].tofile(os.path.join(self.face_db, '%s.jpg' % user_name))
        self.faces_embedding.append({
            "user_name": user_name,
            "feature": embedding
        })
        return "success"

    def detect(self, image):
        faces = self.model.get(image)
        results = list()
        for face in faces:
            result = dict()
            result["bbox"] = np.array(face.bbox).astype(np.int32)
            result["kps"] = np.array(face.kps).astype(np.int32).tolist()
            result["det_score"] = float(face.det_score)
            if face.gender is not None:
                result["gender"] = face.gender
            if face.age is not None:
                result["age"] = face.age
            if face.embedding is not None:
                result["embedding"] = face.embedding
            if face.embedding is not None:
                embedding = np.array(face.embedding).reshape((1, -1))
                embedding = preprocessing.normalize(embedding)
                user_name = "unknown"
                for com_face in self.faces_embedding:
                    if self.feature_compare(embedding, com_face["feature"], self.threshold):
                        user_name = com_face["user_name"]
                        break
                result["name"] = user_name
            else:
                result["name"] = "unknown"
            results.append(result)
        return results


# 第三段：定义图像处理函数

In [3]:
detector = FaceRecognition()

def fun_img(frame):
    out = []
    detection_results = detector.detect(frame)[:1]
    for result in detection_results:
        bbox = result["bbox"]
        kps = result["kps"][:2]
        det_score = result["det_score"]
        name = result["name"]
        out.append(det_score)
        out.append(name)
        out.append(kps)
        out.append(result["gender"])
        cv2.rectangle(frame, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (0, 255, 0), 2)
        if kps is not None:
            for point in kps:
                cv2.circle(frame, tuple(point), 2, (0, 0, 255), 2)
        cv2.putText(frame, f'Score: {det_score:.2f}', (bbox[0], bbox[1] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
        cv2.putText(frame, f'Name: {name}', (bbox[0], bbox[3] + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        if "gender" in result:
            gender = "Male" if result["gender"] == 1 else "Female"
            cv2.putText(frame, gender, (bbox[0], bbox[3] + 40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        if "age" in result:
            age = result["age"]
            cv2.putText(frame, f'Age: {age}', (bbox[0], bbox[3] + 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    return out


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)


# 第四段：主函数和服务器设置

In [4]:
def main():
    host_name = socket.gethostname()
    host_ip = socket.gethostbyname(host_name)
    port = 12345
    host_ip='localhost'
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_socket.bind((host_ip, port))
    server_socket.listen(1)
    print("等待连接...")
    while True:
        conn, addr = server_socket.accept()
        print(f"连接来自: {addr}")
        count=0
        try:
            kpss=[]
            names=[]
            while True:
                count+=1
                print(count)
                    # 接收图片大小
                data = conn.recv(4)
                if not data:
                    break
                img_size = struct.unpack('!I', data)[0]
                # print(img_size)
                if img_size==0:
                    break
                img = receive_image(conn,img_size)
                if img is not None:
                    response=fun_img(img)
                    print(response)
                    if  len(response) >= 3:
                        kpss.append(response[2])
                        names.append(response[1])
                        # print(f"len：{len(kpss)}")
                        if len(kpss)==16:
                            kpss.pop(0)
                            names.pop(0)
                            ab=is_turning(kpss)
                            print(f"数据：{ab}")
                            # 这里条件修改
                            if ab[0]>0.005 and ab[1]>0.05:
                                print(find_most_common_string(names))
                                strfun=find_most_common_string(names)
                                if strfun[1]==1:
                                    print("okokok")
                                    response = strfun[0].encode()
                                    conn.sendall(response)
                                else :
                                    print("waiting1")
                                    response = 'waiting1'.encode()
                                    conn.sendall(response)
                            else:
                                print("waiting2")
                                response = 'waiting2'.encode()
                                conn.sendall(response)

                        else:
                            print("waiting3")
                            response = 'waiting3'.encode()
                            conn.sendall(response)
                    else:
                        print("waiting4")
                        response = 'waiting4'.encode()
                        conn.sendall(response)
                else:
                    print("waiting5")
                    print("没有接收到图片")
                    response = 'waiting5'.encode()
                    conn.sendall(response)
        finally:
            conn.close()
    server_socket.close()
if __name__ == "__main__":
    main()



# netstat -ano | findstr :12345
# taskkill /PID <pid> /F

等待连接...
连接来自: ('127.0.0.1', 12750)
1
[0.8249523639678955, 'caozhexuan (2)', [[224, 238], [332, 232]], 1]
waiting3
2
[0.826187252998352, 'caozhexuan (2)', [[225, 239], [331, 232]], 1]
waiting3
3
[0.8191812038421631, 'caozhexuan (2)', [[227, 238], [335, 232]], 1]
waiting3
4
[0.8191812038421631, 'caozhexuan (2)', [[227, 238], [335, 232]], 1]
waiting3
5
[0.8141299486160278, 'caozhexuan (2)', [[230, 239], [337, 233]], 1]
waiting3
6
[0.8085857629776001, 'caozhexuan (2)', [[230, 240], [338, 233]], 1]
waiting3
7
[0.8061069250106812, 'caozhexuan (2)', [[230, 239], [337, 233]], 1]
waiting3
8
[0.8141416907310486, 'caozhexuan (2)', [[228, 238], [337, 232]], 1]
waiting3
9
[0.7851666212081909, 'caozhexuan (2)', [[230, 218], [344, 210]], 1]
waiting3
10
[0.7851666212081909, 'caozhexuan (2)', [[230, 218], [344, 210]], 1]
waiting3
11
[0.7151220440864563, 'caozhexuan (2)', [[238, 195], [358, 188]], 1]
waiting3
12
[0.7353911399841309, 'lhk (2)', [[41, 64], [92, 61]], 1]
waiting3
13
[0.6363639831542969, 'l